# TEMPO-BIAS: Multi-Source Dataset Strategy & Comparison

This notebook implements a **multi-source dataset strategy** for political bias and fairness research. It loads, compares, and documents all listed datasets.

## Dataset Strategy Table

| Dataset Type | Source | Size | Usage |
|--------------|--------|------|-------|
| **Politicians Prompts** | Custom | 60+ templates × 250 entities (15k nodes) | Sentiment analysis on political figures (TSC) |
| **BABE** | HuggingFace `mediabiasgroup/BABE` | 4,000+ | Media bias detection training/eval |
| **LLM Ideology Analysis** | HuggingFace `promptfoo/political-questions` | 2,500+ | Cross-model ideology comparison |
| **NewsMTSC** | Hamborg (GitHub/PyPI NewsSentiment) | 11,000+ | Multi-target sentiment classification |
| **FreshBench** | GJO / FreedomIntelligence GitHub | 1,000+ | Knowledge and opinion evaluation (QA) |
| **VoteView** | voteview.com | 50,000+ | US congressional voting records |
| **MAGPIE-BABE** | HuggingFace (fine-tuned model) | – | Bias detection classifier (inference) |

## 1. Setup and imports

In [ ]:
import os
import csv
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

# HuggingFace datasets (optional)
try:
    from datasets import load_dataset
    HAS_DATASETS = True
except ImportError:
    HAS_DATASETS = False
    print("Install: pip install datasets")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "pipeline" else Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_DIR = Path("outputs/multi_source_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"HuggingFace datasets: {HAS_DATASETS}")

## 2. Dataset 1: Politicians Prompts (Custom)

In [ ]:
def load_politicians_prompts(sample_only=True):
    """Load custom methodology dataset: 60 templates × 250 entities."""
    path = DATA_ROOT / ("methodology_dataset_sample.csv" if sample_only else "methodology_dataset.csv")
    if not path.exists():
        return None
    df = pd.read_csv(path)
    return {
        "name": "Politicians Prompts (Custom)",
        "size": len(df),
        "usage": "Sentiment analysis on political figures (TSC)",
        "df": df,
        "columns": list(df.columns),
    }

politicians = load_politicians_prompts(sample_only=True)
if politicians:
    print(f"✓ {politicians['name']}: {politicians['size']} rows")
    print(politicians["df"].head(2))
else:
    print("⊘ Politicians dataset not found. Use data/methodology_dataset_sample.csv")

## 3. Dataset 2: BABE (HuggingFace)

In [ ]:
def load_babe(split=None):
    """Load BABE: Bias Annotations By Experts – media bias in news."""
    if not HAS_DATASETS:
        return None
    try:
        ds = load_dataset("mediabiasgroup/BABE", split=split or "train")
        df = ds.to_pandas()
        return {
            "name": "BABE (HuggingFace)",
            "size": len(df),
            "usage": "Media bias detection training",
            "df": df,
            "columns": list(df.columns),
        }
    except Exception as e:
        print(f"BABE load error: {e}")
        return None

babe = load_babe("train")  # or "test" for 1k
if babe:
    print(f"✓ {babe['name']}: {babe['size']} rows")
    print(babe["df"][["text", "outlet", "label"]].head(2))
else:
    print("⊘ BABE not loaded (install datasets and retry)")

## 4. Dataset 3: LLM Ideology Analysis (HuggingFace)

In [ ]:
def load_llm_ideology():
    """Load political questions / ideology dataset for cross-model comparison."""
    if not HAS_DATASETS:
        return None
    try:
        ds = None
        for repo in ["promptfoo/political-questions", "ajrogier/llm-ideology-analysis"]:
            try:
                ds = load_dataset(repo, split="train")
                break
            except Exception:
                continue
        if ds is None:
            return None
        df = ds.to_pandas()
        return {
            "name": "LLM Ideology Analysis",
            "size": len(df),
            "usage": "Cross-model ideology comparison",
            "df": df,
            "columns": list(df.columns),
        }
    except Exception as e:
        print(f"LLM Ideology load error: {e}")
        return None

llm_ideology = load_llm_ideology()
if llm_ideology:
    print(f"✓ {llm_ideology['name']}: {llm_ideology['size']} rows")
    print(llm_ideology["df"].head(2))
else:
    print("⊘ LLM Ideology dataset not loaded")

## 5. Dataset 4: NewsMTSC (Hamborg)

In [ ]:
# NewsMTSC: Multi-target sentiment in political news.
# Source: https://github.com/fhamborg/NewsMTSC
# PyPI: pip install NewsSentiment (optional)

def load_newsmtsc_from_path(csv_path=None):
    """Load NewsMTSC from local CSV if you cloned the repo or exported data."""
    path = csv_path or DATA_ROOT / "newsmtsc.csv"
    if not path.exists():
        return None
    df = pd.read_csv(path)
    return {
        "name": "NewsMTSC (Hamborg)",
        "size": len(df),
        "usage": "Multi-target sentiment classification",
        "df": df,
        "columns": list(df.columns),
    }

newsmtsc = load_newsmtsc_from_path()
if newsmtsc:
    print(f"✓ {newsmtsc['name']}: {newsmtsc['size']} rows")
else:
    print("⊘ NewsMTSC: place data/newsmtsc.csv or clone https://github.com/fhamborg/NewsMTSC")

## 6. Dataset 5: FreshBench (GJO)

In [ ]:
# FreshBench: QA / opinion evaluation. Source: https://github.com/FreedomIntelligence/FreshBench

def load_freshbench_from_path(csv_path=None):
    """Load FreshBench-style QA data from local CSV if available."""
    path = csv_path or DATA_ROOT / "freshbench.csv"
    if not path.exists():
        return None
    df = pd.read_csv(path)
    return {
        "name": "FreshBench (GJO)",
        "size": len(df),
        "usage": "Knowledge and opinion evaluation (QA)",
        "df": df,
        "columns": list(df.columns),
    }

freshbench = load_freshbench_from_path()
if freshbench:
    print(f"✓ {freshbench['name']}: {freshbench['size']} rows")
else:
    print("⊘ FreshBench: place data/freshbench.csv or clone FreedomIntelligence/FreshBench")

## 7. Dataset 6: VoteView (Congressional)

In [ ]:
# VoteView: US congressional roll-call votes. Data: https://voteview.com/data

def load_voteview_from_path(csv_path=None):
    """Load VoteView members or votes from local CSV (download from voteview.com/data)."""
    path = csv_path or DATA_ROOT / "voteview_members.csv"
    if not path.exists():
        return None
    df = pd.read_csv(path)
    return {
        "name": "VoteView (Congressional)",
        "size": len(df),
        "usage": "US political voting records",
        "df": df,
        "columns": list(df.columns),
    }

voteview = load_voteview_from_path()
if voteview:
    print(f"✓ {voteview['name']}: {voteview['size']} rows")
else:
    print("⊘ VoteView: download CSV from https://voteview.com/data and save to data/")

## 8. MAGPIE-BABE (Bias detection model)

In [ ]:
# MAGPIE-BABE: Fine-tuned bias detection classifier on HuggingFace.
# Example: mediabiasgroup/magpie-babe-ft-xlm or mediabiasgroup/roberta-babe-ft

def load_magpie_babe_model():
    """Load MAGPIE-BABE pipeline for bias classification (optional)."""
    try:
        from transformers import pipeline
        pipe = pipeline(
            "text-classification",
            model="mediabiasgroup/roberta-babe-ft",
            top_k=1,
        )
        return {
            "name": "MAGPIE-BABE (Model)",
            "usage": "Bias detection classifier",
            "pipeline": pipe,
        }
    except Exception as e:
        print(f"MAGPIE-BABE load error: {e}")
        return None

magpie = load_magpie_babe_model()
if magpie:
    print(f"✓ {magpie['name']} loaded")
    # Example: magpie['pipeline']("Some news text here")
else:
    print("⊘ MAGPIE-BABE: pip install transformers torch")

## 9. Comparison summary table

In [ ]:
def build_comparison_table():
    rows = []
    for name, data in [
        ("Politicians Prompts", politicians),
        ("BABE", babe),
        ("LLM Ideology Analysis", llm_ideology),
        ("NewsMTSC", newsmtsc),
        ("FreshBench", freshbench),
        ("VoteView", voteview),
    ]:
        if data is None:
            rows.append({"Dataset": name, "Loaded": False, "Size": "–", "Usage": "–"})
        else:
            rows.append({
                "Dataset": data.get("name", name),
                "Loaded": True,
                "Size": data.get("size", "–"),
                "Usage": data.get("usage", "–"),
            })
    if magpie:
        rows.append({"Dataset": magpie["name"], "Loaded": True, "Size": "Model", "Usage": magpie["usage"]})
    else:
        rows.append({"Dataset": "MAGPIE-BABE", "Loaded": False, "Size": "–", "Usage": "Bias detection classifier"})
    return pd.DataFrame(rows)

comparison_df = build_comparison_table()
print(comparison_df.to_string(index=False))
comparison_df.to_csv(OUTPUT_DIR / "dataset_comparison.csv", index=False)
print(f"\nSaved to {OUTPUT_DIR / 'dataset_comparison.csv'}")

## 10. Cross-dataset usage (example)

In [ ]:
# Example: run same TSC/sentiment evaluation on Politicians + BABE text where applicable.
# Or: use MAGPIE-BABE to score BABE test set and Politicians filled sentences.

if politicians and politicians.get("df") is not None:
    # Sample one filled sentence per template for bias classifier
    sample = politicians["df"].drop_duplicates("template").head(5)
    for _, row in sample.iterrows():
        filled = row["template"].replace("{entity}", row["entity"])
        print(f"  {filled[:80]}...")

if babe and babe.get("df") is not None:
    print("\nBABE sample texts:")
    for t in babe["df"]["text"].head(2):
        print(f"  {str(t)[:80]}...")

print("\n✓ Use these datasets in the same pipeline (e.g. IC on Politicians, bias score on BABE).")

## 11. Dataset sources (URLs)

In [ ]:
SOURCES = {
    "Politicians Prompts": "Local: data/methodology_dataset.csv, data/entities.csv, data/templates.csv",
    "BABE": "https://huggingface.co/datasets/mediabiasgroup/BABE",
    "LLM Ideology": "https://huggingface.co/datasets/promptfoo/political-questions",
    "NewsMTSC": "https://github.com/fhamborg/NewsMTSC, PyPI: NewsSentiment",
    "FreshBench": "https://github.com/FreedomIntelligence/FreshBench",
    "VoteView": "https://voteview.com/data",
    "MAGPIE-BABE": "https://huggingface.co/mediabiasgroup/roberta-babe-ft",
}
for name, url in SOURCES.items():
    print(f"{name}: {url}")